In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# 1) EDIT THESE 4 PATHS (one folder per conference)
conference_folders = {
    "Big Ten": r"C:\Users\Dana\OneDrive\Documents\Big 10 Meets",
    "SEC":     r"C:\Users\Dana\OneDrive\Documents\SEC Meets",
    "Big 12":  r"C:\Users\Dana\OneDrive\Documents\Big 12 Meets",
    "ACC":     r"C:\Users\Dana\OneDrive\Documents\ACC Meets",
}

# 2) Column contract (every CSV must have these)
required_cols = [
    "Final Place","Athlete","POINTS",
    "100m","LJ","SP","HJ","400m","110mH","DT","PV","JT","1500m"
]

frames = []

for conf, folder in conference_folders.items():
    folder = str(folder).strip('"')   # handles "Copy as path"
    data_dir = Path(folder)

    files = sorted(data_dir.glob("*.csv"))
    if len(files) == 0:
        raise FileNotFoundError(f"No CSV files found for {conf} in: {data_dir}")

    print(f"{conf}: {len(files)} files")

    for f in files:
        tmp = pd.read_csv(f)

        missing = set(required_cols) - set(tmp.columns)
        if missing:
            raise ValueError(f"{conf} | {f.name} is missing columns: {sorted(missing)}")

        tmp = tmp.copy()
        tmp["conference"] = conf

        # make meet_id unique across conferences (prevents collisions if filenames repeat)
        conf_tag = conf.replace(" ", "").replace("-", "")
        tmp["meet_id"] = f"{conf_tag}_{f.stem}"

        frames.append(tmp)

df_all = pd.concat(frames, ignore_index=True)
df = df_all  # keep later cells working

print("\nRows:", len(df_all), "| Meets:", df_all["meet_id"].nunique())
print("\nMeets by conference:\n", df_all["conference"].value_counts())

df_all[["conference","meet_id","Final Place","Athlete","POINTS"]].head()

In [ ]:
event_cols = ["100m","LJ","SP","HJ","400m","110mH","DT","PV","JT","1500m"]

# 1) Real "meets by conference" (your current print is rows)
print("Unique meets by conference:")
print(df.groupby("conference")["meet_id"].nunique())

# 2) Rows per meet (should be <= 8; not necessarily always 8)
rows_per_meet = df.groupby(["conference","meet_id"]).size().rename("n_rows")
print("\nRows per meet distribution:")
print(rows_per_meet.value_counts().sort_index())

# 3) HARD FAIL if *anything* incomplete/missing exists
bad_tokens = {"DNS","DNF","NH","NT","NM","NP","DQ","NA","-","--",""}

bad = []
for col in event_cols:
    s = df[col]
    ss = s.astype(str).str.strip()

    mask = s.isna() | ss.str.upper().isin(bad_tokens)
    if mask.any():
        tmp = df.loc[mask, ["conference","meet_id","Athlete","Final Place", col]].copy()
        tmp = tmp.rename(columns={col: "bad_value"})
        tmp["event"] = col
        bad.append(tmp)

if bad:
    bad_df = pd.concat(bad, ignore_index=True)
    display(bad_df.head(25))
    raise ValueError(f"STOP: Found {len(bad_df)} invalid event marks (DNS/NH/NT/etc or missing). Fix upstream CSVs.")
else:
    print("\nOK: No DNS/NH/NT/etc and no missing marks in event columns.")

df["overall_score"] = -df["Final Place"]   # higher = better overall finish

In [ ]:
def time_to_seconds_strict(x):
    s = str(x).strip()
    if ":" not in s:
        raise ValueError(f"1500m not in M:SS.ss format: {x}")
    m, sec = s.split(":")
    return int(m) * 60 + float(sec)

df["1500m_sec"] = df["1500m"].apply(time_to_seconds_strict)

print("1500m_sec created. Example rows:")
display(df[["meet_id","Athlete","1500m","1500m_sec"]].head(10))

In [ ]:
df_rank = df.copy()

event_cols = ["100m","LJ","SP","HJ","400m","110mH","DT","PV","JT"]
num_cols = [c + "_num" for c in event_cols]

# 1) Make numeric versions of all event marks (extract first number from strings like "7.12", "7.12m", etc.)
for c in event_cols:
    s = df_rank[c].astype(str).str.strip()
    extracted = s.str.extract(r"([0-9]*\.?[0-9]+)")[0]
    df_rank[c + "_num"] = pd.to_numeric(extracted, errors="coerce")

# 2) HARD FAIL if anything didn't parse (should be zero because Cell 2 already validated)
bad_parse = df_rank[df_rank[num_cols].isna().any(axis=1)][
    ["conference","meet_id","Athlete","Final Place"] + event_cols + num_cols
]
if len(bad_parse) > 0:
    display(bad_parse.head(15))
    raise ValueError(f"STOP: {len(bad_parse)} rows have non-numeric event marks after parsing. Fix upstream CSV(s).")

# 3) Rank within each meet (use numeric columns only)
g = df_rank.groupby("meet_id")

# track (lower is better)
df_rank["rank_100m"]  = g["100m_num"].rank(ascending=True, method="min")
df_rank["rank_400m"]  = g["400m_num"].rank(ascending=True, method="min")
df_rank["rank_110H"]  = g["110mH_num"].rank(ascending=True, method="min")
df_rank["rank_1500m"] = g["1500m_sec"].rank(ascending=True, method="min")  # from Cell 3

# field (higher is better)
df_rank["rank_LJ"] = g["LJ_num"].rank(ascending=False, method="min")
df_rank["rank_SP"] = g["SP_num"].rank(ascending=False, method="min")
df_rank["rank_HJ"] = g["HJ_num"].rank(ascending=False, method="min")
df_rank["rank_DT"] = g["DT_num"].rank(ascending=False, method="min")
df_rank["rank_PV"] = g["PV_num"].rank(ascending=False, method="min")
df_rank["rank_JT"] = g["JT_num"].rank(ascending=False, method="min")

display(df_rank[["meet_id","Final Place","Athlete",
         "rank_100m","rank_LJ","rank_SP","rank_HJ","rank_400m","rank_110H",
         "rank_DT","rank_PV","rank_JT","rank_1500m"]].head(10))

In [ ]:
rank_cols = [c for c in df_rank.columns if c.startswith("rank_")]

spearman_rank = (
    (-df_rank[rank_cols])  # flip so higher = better
    .corrwith(df_rank["overall_score"], method="spearman")
    .rename("spearman_rho")
    .reset_index()
    .rename(columns={"index": "event"})
)

spearman_rank["event"] = spearman_rank["event"].str.replace("^rank_", "", regex=True)
spearman_rank = spearman_rank.sort_values("spearman_rho", ascending=False)

spearman_rank


In [ ]:
df_z = df_rank.copy()  # reuse *_num columns created in Cell 4

# make sure the numeric columns exist
needed = ["100m_num","400m_num","110mH_num","1500m_sec",
          "LJ_num","SP_num","HJ_num","DT_num","PV_num","JT_num"]
missing = [c for c in needed if c not in df_z.columns]
if missing:
    raise ValueError(f"Missing required numeric columns: {missing}. Run Cell 4 first.")

def z_within(s):
    sd = s.std(ddof=0)
    if sd == 0:
        return s*0.0  # all equal -> everyone gets z = 0
    return (s - s.mean()) / sd

g = df_z.groupby("meet_id")

# track: lower time is better -> flip sign first so higher = better
df_z["z_100m"]  = g["100m_num"].transform(lambda s: z_within(-s))
df_z["z_400m"]  = g["400m_num"].transform(lambda s: z_within(-s))
df_z["z_110H"]  = g["110mH_num"].transform(lambda s: z_within(-s))
df_z["z_1500m"] = g["1500m_sec"].transform(lambda s: z_within(-s))

# field: higher mark is better
df_z["z_LJ"] = g["LJ_num"].transform(z_within)
df_z["z_SP"] = g["SP_num"].transform(z_within)
df_z["z_HJ"] = g["HJ_num"].transform(z_within)
df_z["z_DT"] = g["DT_num"].transform(z_within)
df_z["z_PV"] = g["PV_num"].transform(z_within)
df_z["z_JT"] = g["JT_num"].transform(z_within)

z_cols = [c for c in df_z.columns if c.startswith("z_")]

print("Z-score columns:", z_cols)
display(df_z[["meet_id","Final Place","Athlete"] + z_cols].head())

In [ ]:
pearson_z = (
    df_z[z_cols]
    .corrwith(df_z["overall_score"], method="pearson")
    .rename("pearson_r")
    .reset_index()
    .rename(columns={"index": "event"})
)

pearson_z["event"] = pearson_z["event"].str.replace("^z_", "", regex=True)
pearson_z = pearson_z.sort_values("pearson_r", ascending=False)

pearson_z

In [ ]:
# RQ3 cross-check: Spearman by conference using ranks (uniform "higher is better")
rank_cols = [c for c in df_rank.columns if c.startswith("rank_")]

corr_list = []
for conf, d in df_rank.groupby("conference"):
    s = (-d[rank_cols]).corrwith(d["overall_score"], method="spearman")
    s.name = conf
    corr_list.append(s)

rank_wide = pd.concat(corr_list, axis=1)
rank_wide.index = rank_wide.index.str.replace("^rank_", "", regex=True)
rank_wide["mean_rho"]  = rank_wide.mean(axis=1)
rank_wide["range_rho"] = rank_wide.max(axis=1) - rank_wide.min(axis=1)

display(rank_wide.sort_values("mean_rho", ascending=False))
display(rank_wide.sort_values("range_rho", ascending=False).head(10))

In [ ]:
# sample sizes
conf_sizes = (
    df_z.groupby("conference")
        .agg(n_rows=("overall_score", "size"),
             n_meets=("meet_id", "nunique"))
)
display(conf_sizes)

# correlations by conference -> wide table
corr_list = []
for conf, d in df_z.groupby("conference"):
    s = d[z_cols].corrwith(d["overall_score"], method="pearson")
    s.name = conf
    corr_list.append(s)

z_wide = pd.concat(corr_list, axis=1)
z_wide.index = z_wide.index.str.replace("^z_", "", regex=True)

# summary columns for RQ3
z_wide["mean_r"]  = z_wide.mean(axis=1)
z_wide["range_r"] = z_wide.max(axis=1) - z_wide.min(axis=1)

# show: strongest overall + most conference-different
display(z_wide.sort_values("mean_r", ascending=False))
display(z_wide.sort_values("range_r", ascending=False).head(10))

In [ ]:
plot_df = spearman_rank.copy().sort_values("spearman_rho", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(plot_df["event"], plot_df["spearman_rho"])
plt.xlabel("Spearman ρ with overall_score (higher = stronger alignment)")
plt.title("Figure A: (RQ1) Event rank alignment with overall placing")
plt.tight_layout()
plt.show()


In [ ]:
plot_df = pearson_z.copy().sort_values("pearson_r", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(plot_df["event"], plot_df["pearson_r"], color="orange")
plt.xlabel("Pearson r with overall_score (higher = stronger alignment)")
plt.title("Figure B: (RQ2) Event z-score alignment with overall placing")
plt.tight_layout()
plt.show()


In [ ]:
plot_df = z_wide["range_r"].sort_values(ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(plot_df.index, plot_df.values, color="maroon")
plt.xlabel("Range across conferences (bigger = more different)")
plt.title("Figure C: (RQ3) Conference-to-conference variability by event (z-score method)")
plt.tight_layout()
plt.show()


In [ ]:
plot_df = rank_wide["range_rho"].sort_values(ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(plot_df.index, plot_df.values)
plt.xlabel("Range across conferences (bigger = more different)")
plt.title("Figure D: (RQ3) Conference-to-conference variability by event (rank method)")
plt.tight_layout()
plt.show()

In [ ]:
confs = ["ACC", "Big 12", "Big Ten", "SEC"]

M = z_wide[confs].copy()
M = M.loc[M.mean(axis=1).sort_values(ascending=False).index]  # order events by overall strength

plt.figure(figsize=(7, 6))
img = plt.imshow(M.values, aspect="auto")
plt.colorbar(img, label="Pearson r with overall_score")

plt.xticks(range(len(confs)), confs, rotation=30, ha="right")
plt.yticks(range(len(M.index)), M.index)
plt.title("RQ3: Event importance by conference (z-score method)")
plt.tight_layout()
plt.show()

In [ ]:
# Build meet-level correlations: for each meet_id, correlate each z_* with overall_score
meet_corrs = []
for mid, d in df_z.groupby("meet_id"):
    row = {"meet_id": mid, "conference": d["conference"].iloc[0]}
    for ev in z_cols:
        row[ev.replace("z_", "")] = d[ev].corr(d["overall_score"], method="pearson")
    meet_corrs.append(row)

meet_corrs = pd.DataFrame(meet_corrs)

# Order events by average meet-level correlation
events = [c for c in meet_corrs.columns if c not in ["meet_id","conference"]]
order = meet_corrs[events].mean().sort_values(ascending=False).index.tolist()

plt.figure(figsize=(9, 5))
plt.boxplot([meet_corrs[e].dropna() for e in order], tick_labels=order, vert=False)
plt.xlabel("Meet-level Pearson r with overall_score (higher = stronger alignment)")
plt.title("Meet-to-meet variability of event importance (z-score method)")
plt.tight_layout()
plt.show()